# Solar Filament Segmentation Challenge 2026 -- ResNet50 BYOL Pretraining (Kaggle)

Domain-specific self-supervised pretraining of a 1-channel ResNet50 on the GONG
H-alpha corpus, via BYOL. Runs `scripts/pretrain_resnet/train_byol.py`, DDP-ready
for Kaggle's 2xT4. Design: `RESNET_PRETRAIN_PLAN.md`. Prerequisite: the
`halpha-preprocessed` Kaggle Dataset (built by `pretrain_gong_kaggle.ipynb`) added
as a notebook input, and a GPU accelerator (ideally 2xT4) enabled.

This is a **multi-session** run -- see section 6 below for the checklist to repeat
each session (mount latest checkpoint, run, version a new checkpoint dataset,
repeat).

### 0. Clone the repo

Requires `jp-pretraining-data-prep` (or wherever this lands) to already be pushed
to `origin`.

In [ ]:
!git clone -b jp-pretraining-data-prep https://github.com/jprakash-1/Solar-Filament-Segmentation.git


In [ ]:
%cd Solar-Filament-Segmentation
!ls


In [ ]:
# re-run-safe: pick up any commits pushed after the kernel started
!git pull origin jp-pretraining-data-prep


### 1. Install dependencies

Kaggle's GPU images already ship torch/torchvision/pandas/tqdm -- only add what's
missing (`scipy`, for the augmentation's Gaussian blur; `segmentation_models_pytorch`
only if re-running `scripts/pretrain_resnet/model.py`'s `verify_stem_averaging`
check, not needed for training itself).

In [ ]:
!pip install -q scipy


### 2. Mount the preprocessed corpus (+ a checkpoint dataset, on resume sessions)

In the Kaggle UI: **Add Input -> Datasets -> `halpha-preprocessed`** (first
session and every session), and on any session after the first, also add the
checkpoint dataset from the previous session's `/kaggle/working/checkpoints/`
version (see section 6).

Verify the actual mounted layout before trusting a path below -- it depends on how
the dataset was uploaded/zipped, and guessing wrong here just wastes a session:

In [ ]:
!ls /kaggle/input/
!ls /kaggle/input/halpha-preprocessed/ | head


In [ ]:
# Adjust these two if the listing above shows a different layout
# (e.g. files nested one directory deeper than expected).
IMAGES_DIR = "/kaggle/input/halpha-preprocessed"
MANIFEST = "/kaggle/input/halpha-preprocessed/manifest.csv"

# Point at the mounted checkpoint dataset's latest.pt on resume sessions; leave
# as None (the script's own --resume default) for the very first session.
RESUME_CKPT = None  # e.g. "/kaggle/input/halpha-byol-ckpt/latest.pt"

# Optional: stop the run once val_loss hasn't improved for this many health
# checks (--health-check-every apart, so 10 * 5 = 50 epochs by default) --
# leave as None to always run the full --epochs on its cosine schedule.
EARLY_STOP_PATIENCE = None  # e.g. 10


### 3. Smoke test before the real run

Small subset, single-process, 1 epoch -- catches a broken path or a shape bug in
under a minute, before committing a multi-hour session to it. Matches the
"verify against real data before trusting it at scale" standard the Stage 1/2
scripts were held to.

In [ ]:
!python scripts/pretrain_resnet/train_byol.py \
    --images-dir $IMAGES_DIR --manifest $MANIFEST \
    --max-images 200 --epochs 1 --batch-size 16 --num-workers 2 \
    --checkpoint-out /kaggle/working/smoke_ckpt.pt --log-csv /kaggle/working/smoke_log.csv \
    --nn-grid-out /kaggle/working/smoke_nn.png


### 4. Full run: multi-GPU launch

`torchrun --nproc_per_node=2` for Kaggle's 2xT4 (drop to 1 if only a single GPU is
enabled). `NCCL_P2P_DISABLE=1` is the same T4-specific workaround
`PRETRAIN_PLAN.md` section 4.3 already had to apply. `--session-budget-seconds`
should stay well under the account's actual session cap (varies by verification
tier) -- the script self-stops with margin for the final checkpoint write, no need
to watch the clock.

Writes two checkpoints to `/kaggle/working/checkpoints/`: `latest.pt` (every
epoch -- what `RESUME_CKPT` above should point at) and `best.pt` (only on a
val_loss improvement, at each health check). If `EARLY_STOP_PATIENCE` is set,
the run also self-stops once val_loss plateaus for that many health checks.

In [ ]:
RESUME_ARGS = f"--resume {RESUME_CKPT}" if RESUME_CKPT else ""
EARLY_STOP_ARGS = f"--early-stop-patience {EARLY_STOP_PATIENCE}" if EARLY_STOP_PATIENCE else ""

!NCCL_P2P_DISABLE=1 torchrun --nproc_per_node=2 scripts/pretrain_resnet/train_byol.py \
    --images-dir $IMAGES_DIR --manifest $MANIFEST \
    --epochs 150 --batch-size 128 --num-workers 4 \
    --checkpoint-out /kaggle/working/checkpoints/latest.pt \
    --best-checkpoint-out /kaggle/working/checkpoints/best.pt \
    --log-csv /kaggle/working/checkpoints/loss_log.csv \
    --nn-grid-out /kaggle/working/checkpoints/nn_grid.png \
    --session-budget-seconds 28800 --health-check-every 5 \
    {RESUME_ARGS} {EARLY_STOP_ARGS}


### 5. Sanity-check this session's output

Loss curve, plus the section 8 embedding-std collapse check and nearest-neighbor
visual check that `health_checks.py` already logged during training -- don't
trust the loss curve alone (BYOL can keep dropping loss while representations
collapse).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv("/kaggle/working/checkpoints/loss_log.csv")
print(log.tail())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log["epoch"], log["train_loss"], label="train")
axes[0].plot(log["epoch"], log["val_loss"], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BYOL loss"); axes[0].legend()

health = log.dropna(subset=["embedding_std"])
axes[1].plot(health["epoch"], health["embedding_std"])
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("embedding_std (collapse check)")
plt.tight_layout(); plt.show()


In [ ]:
from PIL import Image
Image.open("/kaggle/working/checkpoints/nn_grid.png")


### 6. Session checklist (repeat each session)

1. Mount `halpha-preprocessed` and the latest `halpha-byol-ckpt` dataset (if any)
   as notebook inputs; set `RESUME_CKPT` above accordingly.
2. Run the smoke test (section 3), then the full launch cell (section 4).
3. The loop self-stops at `--session-budget-seconds` (or once `EARLY_STOP_PATIENCE`
   health checks pass with no val_loss improvement, if set), checkpointing every
   epoch along the way -- an unplanned kill loses at most one epoch.
4. **New Dataset version** from `/kaggle/working/checkpoints/` -> this becomes
   next session's `halpha-byol-ckpt` input. Both `latest.pt` (resume from this)
   and `best.pt` (best val_loss so far) go along for the ride.
5. Run section 5's sanity check before ending the session -- catch a collapsed
   run early rather than after several more sessions of wasted compute.
6. Every few sessions: inspect `nn_grid.png` closely (section 5) -- confirm
   neighbors are astronomically similar, not random frames.

**Open items this run should help resolve** (`RESNET_PRETRAIN_PLAN.md` section
11): real images/sec on 2xT4 at this batch size (watch the tqdm it/s from
section 3/4 and back out a per-GPU throughput number), and whether the full
49K-image corpus vs. some thinning is the right call once real wall-clock/epoch
is known.

### 7. Export the encoder + standalone inference check

`train_byol.py`'s checkpoint (`best.pt`/`latest.pt`) carries the full BYOL
module -- online *and* target encoder/projector/predictor, optimizer, GradScaler.
Stage 4 segmentation fine-tuning only wants the trained `online_encoder`'s
weights. `scripts/pretrain_resnet/export_encoder.py` strips everything else down
to a plain `encoder_state_dict` checkpoint.

`scripts/pretrain_resnet/inference.py` then exercises *that* exported checkpoint
standalone (no BYOL scaffolding loaded, just the plain `resnet50_1ch` backbone --
the way Stage 4 will actually consume it): embeds a fresh random sample of
frames with a deterministic disk-centered crop (no training-time augmentation)
and renders a nearest-neighbor grid, same visual sanity check as section 5/8 but
run independently of the training loop and its held-out val split.

In [ ]:
import os

# Point at whichever training checkpoint you want to export -- best.pt (lowest
# val_loss so far) unless there's a specific reason to prefer the very latest
# epoch instead.
TRAIN_CHECKPOINT = "/kaggle/working/checkpoints/best.pt"
ENCODER_OUT = "/kaggle/working/resnet50_byol_encoder.pt"

# Fails loudly here with a clear message rather than letting a missing/empty
# path surface later as a cryptic argparse error in the shell cell below --
# best.pt only exists once training has passed its first health check
# (section 4), so this is the most likely thing to be missing on a fresh kernel.
assert os.path.exists(TRAIN_CHECKPOINT), (
    f"missing checkpoint: {TRAIN_CHECKPOINT} -- run section 4's training cell "
    "first (best.pt is written at the first --health-check-every health check, "
    "not immediately), or point TRAIN_CHECKPOINT at latest.pt instead."
)

!python scripts/pretrain_resnet/export_encoder.py \
    --checkpoint $TRAIN_CHECKPOINT --out $ENCODER_OUT


In [ ]:
import os

# Guards against the exact failure mode that produces a cryptic
# "--encoder-checkpoint: expected one argument" error below: ENCODER_OUT
# undefined/empty because the export cell above wasn't run in this kernel
# session (e.g. after a Kaggle kernel restart that re-ran only this cell).
assert "ENCODER_OUT" in dir() and os.path.exists(ENCODER_OUT), (
    "ENCODER_OUT is missing or doesn't exist -- run the export-encoder cell "
    "above first (in this kernel session)."
)

!python scripts/pretrain_resnet/inference.py \
    --encoder-checkpoint $ENCODER_OUT \
    --images-dir $IMAGES_DIR --manifest $MANIFEST \
    --num-images 64 --k 5 \
    --grid-out /kaggle/working/inference_nn_grid.png \
    --embeddings-out /kaggle/working/inference_embeddings.npy


In [ ]:
from PIL import Image
Image.open("/kaggle/working/inference_nn_grid.png")


Two more views of the same embeddings, complementary to the neighbor grid above:

- **PCA scatter**, colored by capture date -- the neighbor grid only shows *local*
  structure (is each query's nearest neighbor sensible); this shows *global*
  structure across the whole sample. Smooth date-related structure (if the
  sample spans enough calendar time) is a good sign; a single tight blob is the
  same collapse concern section 5's `embedding_std` plot already watches for,
  seen from a different angle.
- **Pairwise cosine-similarity histogram** -- a collapsed encoder pushes every
  embedding toward the same direction, so similarities bunch up near 1.0; a
  healthy encoder spreads them out. Directly checks the failure mode
  `health_checks.embedding_std` is designed to catch, on this fresh standalone
  sample rather than the training val split.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

embeddings = np.load("/kaggle/working/inference_embeddings.npy")
meta = pd.read_csv("/kaggle/working/inference_embeddings.csv")
months = pd.to_datetime(meta["date"], format="%Y%m%d").dt.month

coords = PCA(n_components=2).fit_transform(embeddings)

plt.figure(figsize=(6, 5))
sc = plt.scatter(coords[:, 0], coords[:, 1], c=months, cmap="viridis", s=40)
plt.colorbar(sc, label="capture month")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("Inference-sample embeddings, PCA to 2D")
plt.tight_layout(); plt.show()


In [ ]:
# embeddings are already L2-normalized (inference.py applies F.normalize), so a
# plain dot product is cosine similarity -- only the upper triangle, excluding
# the diagonal, so each pair is counted once and self-similarity (always 1.0)
# doesn't dominate the histogram.
sims = embeddings @ embeddings.T
iu = np.triu_indices_from(sims, k=1)
pairwise_sims = sims[iu]

plt.figure(figsize=(6, 4))
plt.hist(pairwise_sims, bins=40, range=(-1, 1))
plt.xlabel("cosine similarity"); plt.ylabel("pair count")
plt.title("Pairwise embedding similarity (collapse check)")
plt.tight_layout(); plt.show()
print(f"mean={pairwise_sims.mean():.3f} std={pairwise_sims.std():.3f}")
